In [32]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pyomo.environ as pyo
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from pull_prices import merged_df_clean, merged_df_spike
from params import nodes, mcp, mdp, e, fee

# Create output folder
BASE_DIR = os.getcwd()
PLOT_DIR = os.path.join(BASE_DIR, "results", "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

print("Plots will be saved in:", PLOT_DIR)

Plots will be saved in: d:\battery-storage-optimization-energy-ancillary\results\plots


In [33]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 32)
        self.tr = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(32, 4, batch_first=True), 2
        )
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        return self.out(self.tr(self.fc(x)))


def train_transformer(df):

    if df is None:
        raise ValueError("train_transformer received None")

    df = df.copy()

    if "SP15" not in df.columns:
        raise KeyError("SP15 column missing")

    # EWMA-based expected behavior
    df["expected_price"] = df["SP15"].ewm(span=24, adjust=False).mean()

    return df

In [34]:
def prepare_timeseries(df):

    # average price across nodes (simple fix)
    df_ts = df.groupby("datetime").agg({
        "SP15": "mean",
        "NonSpin": "mean",
        "RegDown": "mean",
        "RegUp": "mean",
        "Spin": "mean"
    }).reset_index()

    df_ts = df_ts.sort_values("datetime").reset_index(drop=True)

    return df_ts

In [35]:
# =========================
# 🔹 GRAPH (LEARNED)
# =========================
def build_graph(df):

    pivot = df.pivot(index="datetime", columns="node", values="SP15")
    pivot = pivot.ffill()

    A = pivot.corr().values
    np.fill_diagonal(A, 0)

    D = np.diag(A.sum(axis=1))
    L = D - A

    return L

In [36]:
def compute_anomaly(df):
    df = df.copy()

    if "expected_price" not in df.columns:
        raise KeyError("expected_price missing — run train_vae first")

    X = df[["SP15"]].values

    iso = IsolationForest(contamination=0.05)
    lof = LocalOutlierFactor()

    df["iso"] = (iso.fit_predict(X) == -1).astype(int)
    df["lof"] = (lof.fit_predict(X) == -1).astype(int)

    df["res"] = abs(df["SP15"] - df["expected_price"])

    scaler = MinMaxScaler()
    df[["res"]] = scaler.fit_transform(df[["res"]])

    df["anomaly"] = df["res"] + df["iso"] + df["lof"]
    df["anomaly"] = MinMaxScaler().fit_transform(df[["anomaly"]])

    return df

In [37]:
def classify_anomaly(df, window=6, persist_threshold=2):
    df = df.copy()

    # rate of change
    df["roc"] = df["SP15"].diff().abs()
    df["roc"] = df["roc"] / (df["SP15"].shift(1).abs() + 1e-8)
    df["roc"] = df["roc"].fillna(0)  # row 0 has no prior — treat as no change

    # persistence — how many surrounding hours are also anomalous
    df["anom_binary"]  = (df["anomaly"] > 0.5).astype(int)
    df["persistence"]  = (
        df["anom_binary"]
        .rolling(window, center=True, min_periods=1)
        .sum()
    )

    # local z-score
    df["local_mean"] = df["SP15"].rolling(window, center=True, min_periods=1).mean()
    df["local_std"]  = df["SP15"].rolling(window, center=True, min_periods=1).std().fillna(1)
    df["z_score"]    = (df["SP15"] - df["local_mean"]) / df["local_std"]

    # mean reversion — does price come back down after the spike?
    df["future_mean"] = df["SP15"].shift(-window).rolling(window, min_periods=1).mean()
    df["reverts"]     = (df["future_mean"] < df["SP15"]).astype(int)

    # genuine if: persists + reverts + not an instant jump
    df["is_genuine"] = (
        (df["persistence"] >= persist_threshold) &
        (df["reverts"] == 1) &
        (df["roc"] < 0.5)
    ).astype(int)

    # boost genuine spikes, suppress noise
    df["anomaly_adjusted"] = df["anomaly"] * np.where(df["is_genuine"], 1.5, 0.3)
    df["anomaly_adjusted"] = df["anomaly_adjusted"].clip(0, 1)

    return df

In [38]:
def optimize(df, L, mode="baseline"):
    """
    mode: 
      'baseline'      — no anomaly awareness
      'penalised'     — avoids all anomalous hours
      'opportunistic' — exploits trending anomalies, avoids noise
    """
    df = df.copy().reset_index(drop=True)
    df["anomaly"]    = df["anomaly"].fillna(0).clip(0, 1)
    df["is_genuine"] = df["is_genuine"].fillna(0)

    anomaly_sensitivity = 0.3

    model = pyo.ConcreteModel()
    T = len(df)

    model.t = pyo.RangeSet(0, T - 1)
    model.n = pyo.Set(initialize=nodes)

    model.price    = pyo.Param(model.t, initialize=lambda m, t: float(df["SP15"].iloc[t]))
    model.anom     = pyo.Param(model.t, initialize=lambda m, t: float(df["anomaly"].iloc[t]))
    model.genuine  = pyo.Param(model.t, initialize=lambda m, t: float(df["is_genuine"].iloc[t]))

    model.buy  = pyo.Var(model.t, model.n, bounds=(0, mcp))
    model.sell = pyo.Var(model.t, model.n, bounds=(0, 1.2 * mdp))
    model.soc  = pyo.Var(model.t, bounds=(0, mcp))

    # --- SOC BALANCE ---
    def soc_rule(m, t):
        if t == 0:
            return m.soc[t] == mcp
        return m.soc[t] == m.soc[t-1] \
            + sum(m.buy[t, n] for n in m.n) * e \
            - sum(m.sell[t, n] for n in m.n) / e

    model.soc_c = pyo.Constraint(model.t, rule=soc_rule)

    # --- PHYSICAL FEASIBILITY ---
    def sell_energy_rule(m, t):
        if t == 0:
            return sum(m.sell[t, n] for n in m.n) == 0
        return sum(m.sell[t, n] for n in m.n) / e <= m.soc[t - 1]

    model.sell_energy_c = pyo.Constraint(model.t, rule=sell_energy_rule)

    def buy_headroom_rule(m, t):
        if t == 0:
            return pyo.Constraint.Skip
        return sum(m.buy[t, n] for n in m.n) * e <= mcp - m.soc[t - 1]

    model.buy_headroom_c = pyo.Constraint(model.t, rule=buy_headroom_rule)

    # --- SELL CAP (mode-dependent) ---
    def sell_limit_rule(m, t, n):
        if mode == "baseline":
            return m.sell[t, n] <= mdp

        elif mode == "penalised":
            # shrink sell cap during ANY anomaly
            return m.sell[t, n] <= mdp * (1 - anomaly_sensitivity * m.anom[t])

        elif mode == "opportunistic":
            # expand cap for genuine trending anomalies
            # restrict cap for noise/manipulation
            return m.sell[t, n] <= mdp * (
                1
                + anomaly_sensitivity * m.anom[t] * m.genuine[t]
                - anomaly_sensitivity * m.anom[t] * (1 - m.genuine[t])
            )

    model.sell_limit = pyo.Constraint(model.t, model.n, rule=sell_limit_rule)

    # --- OBJECTIVE (mode-dependent) ---
    def obj(m):
        if mode == "baseline":
            return sum(
                m.sell[t, n] * m.price[t]
                - m.buy[t, n] * m.price[t]
                - fee * (m.sell[t, n] + m.buy[t, n])
                for t in m.t for n in m.n
            )

        elif mode == "penalised":
            # discourage all trading during anomalies
            return sum(
                m.sell[t, n] * m.price[t] * (1 - anomaly_sensitivity * m.anom[t])
                - m.buy[t, n] * m.price[t] * (1 + anomaly_sensitivity * m.anom[t])
                - fee * (m.sell[t, n] + m.buy[t, n])
                for t in m.t for n in m.n
            )

        elif mode == "opportunistic":
            # exploit genuine trending spikes aggressively
            # penalise buying/selling into noise
            return sum(
                m.sell[t, n] * m.price[t] * (
                    1 + anomaly_sensitivity * m.anom[t] * m.genuine[t]       # boost genuine sell
                    - anomaly_sensitivity * m.anom[t] * (1 - m.genuine[t])   # suppress noise sell
                )
                - m.buy[t, n] * m.price[t] * (
                    1 - anomaly_sensitivity * m.anom[t] * m.genuine[t]       # cheap buy before genuine spike
                    + anomaly_sensitivity * m.anom[t] * (1 - m.genuine[t])   # penalise buying into noise
                )
                - fee * (m.sell[t, n] + m.buy[t, n])
                for t in m.t for n in m.n
            )

    model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

    solver_result = pyo.SolverFactory("highs").solve(model)

    if solver_result.solver.termination_condition != pyo.TerminationCondition.optimal:
        print(f"[WARNING] Solver termination: {solver_result.solver.termination_condition}")

    soc    = [pyo.value(model.soc[t]) for t in model.t]
    profit = pyo.value(model.obj) or 0.0

    return profit, soc

In [39]:
# =========================
# 🔹 PLOTTING (SAVE ONLY)
# =========================

def plot_price_comparison(df, name):

    plt.figure(figsize=(12,5))
    plt.plot(df["SP15"].values, label="Actual Price")
    plt.plot(df["expected_price"].values, label="Expected Price")

    plt.title("Price vs Expected Price")
    plt.legend()
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_price.png")
    plt.savefig(filepath)
    plt.close()


def plot_soc_comparison(base_soc, anom_soc, name):

    plt.figure(figsize=(12,5))
    plt.plot(base_soc, label="Baseline SOC")
    plt.plot(anom_soc, label="Anomaly SOC")

    plt.title("SOC Comparison")
    plt.legend()
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_soc.png")
    plt.savefig(filepath)
    plt.close()


def plot_volatility(df, name):

    df = df.copy()
    df["volatility"] = df["SP15"].rolling(24).std()

    plt.figure(figsize=(12,4))
    plt.plot(df["volatility"], color="purple")

    plt.title("Volatility")
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_volatility.png")
    plt.savefig(filepath)
    plt.close()


def plot_profit(c_base, c_pen, c_opp, a_base, a_pen, a_opp):
    x     = np.arange(3)
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width, [c_base, c_pen, c_opp], width, label="Clean",  color="steelblue")
    ax.bar(x,         [a_base, a_pen, a_opp], width, label="Attack", color="crimson")

    ax.set_xticks(x)
    ax.set_xticklabels(["Baseline", "Penalised", "Opportunistic"])
    ax.set_title("Profit Comparison Across Strategies and Scenarios")
    ax.set_ylabel("Profit ($)")
    ax.legend()
    ax.grid(axis="y")

    filepath = os.path.join(PLOT_DIR, "profit_comparison.png")
    plt.savefig(filepath)
    plt.close()

In [40]:
class VAE(nn.Module):
    def __init__(self, input_dim=1, latent_dim=8):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, 16)
        self.fc3 = nn.Linear(16, input_dim)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return self.fc3(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [41]:
def train_vae(df):
    # NOTE: Transformer defined but not used in pipeline.
    # train_vae (VAE) is used instead. Keep for future experimentation.

    df = df.copy()

    data = df["SP15"].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)

    x = torch.tensor(data_scaled, dtype=torch.float32)

    model = VAE()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    def loss_fn(recon, x, mu, logvar):
        recon_loss = nn.MSELoss()(recon, x)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon_loss + kl_loss

    # Training
    dataset = torch.utils.data.TensorDataset(x)
    loader  = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

    model.train()
    for epoch in range(50):
        epoch_loss = 0.0
        for (batch,) in loader:
            optimizer.zero_grad()
            recon, mu, logvar = model(batch)
            loss = loss_fn(recon, batch, mu, logvar)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"[VAE] Epoch {epoch+1}/50 — Loss: {epoch_loss:.4f}")

    # Reconstruction (outside training loop)
    model.eval()
    with torch.no_grad():
        recon, _, _ = model(x)
    recon_np = recon.numpy()

    df["expected_price"] = scaler.inverse_transform(recon_np)

    return df, model, scaler   # ← model and scaler needed for Grad-CAM in run()

In [42]:
def compute_gradcam(model, x_tensor, target_idx=None):
    """
    Compute Grad-CAM scores for VAE reconstruction.
    Returns a numpy array of shape (T,) — one score per timestep.
    Higher score = this timestep had more influence on reconstruction error.
    """
    model.eval()

    # We'll compute gradient of reconstruction loss w.r.t. input
    x_input = x_tensor.clone().requires_grad_(True)

    recon, mu, logvar = model(x_input)

    # Reconstruction loss per timestep (not reduced)
    recon_loss_per_t = (recon - x_input).pow(2).squeeze()  # shape (T,)

    if target_idx is not None:
        # Gradient w.r.t. a specific timestep's loss
        scalar = recon_loss_per_t[target_idx]
    else:
        # Gradient w.r.t. total reconstruction loss
        scalar = recon_loss_per_t.sum()

    scalar.backward()

    # Gradient of loss w.r.t. input — shape (T, 1)
    gradients = x_input.grad.abs().squeeze().detach().numpy()  # shape (T,)

    # Normalise to [0, 1]
    gradients = (gradients - gradients.min()) / (gradients.max() - gradients.min() + 1e-8)

    return gradients


def plot_gradcam(df, gradcam_scores, name):
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    axes[0].plot(df["SP15"].values, label="SP15 Price", color="steelblue")
    axes[0].plot(df["expected_price"].values, label="Expected Price", color="orange", linestyle="--")
    axes[0].set_title("Price vs Expected")
    axes[0].legend()
    axes[0].grid()

    axes[1].bar(range(len(gradcam_scores)), gradcam_scores, color="crimson", alpha=0.7)
    axes[1].set_title("Grad-CAM: VAE Attention per Timestep")
    axes[1].set_xlabel("Timestep (hours)")
    axes[1].set_ylabel("Influence Score")
    axes[1].grid(axis="y")

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, f"{name}_gradcam.png")
    plt.savefig(filepath)
    plt.close()
    print(f"[GradCAM] Saved: {filepath}")

In [43]:
def run(df):
    df_graph = df.copy()
    df_ts = prepare_timeseries(df)

    df_ts, vae_model, vae_scaler = train_vae(df_ts)
    df_ts = compute_anomaly(df_ts)
    df_ts = classify_anomaly(df_ts)
    df_ts["anomaly"] = df_ts["anomaly_adjusted"]

    data_scaled      = vae_scaler.transform(df_ts["SP15"].values.reshape(-1, 1))
    x_tensor         = torch.tensor(data_scaled, dtype=torch.float32)
    gradcam          = compute_gradcam(vae_model, x_tensor)
    df_ts["gradcam"] = gradcam

    L = build_graph(df_graph)

    base_profit,  base_soc  = optimize(df_ts, L, mode="baseline")
    pen_profit,   pen_soc   = optimize(df_ts, L, mode="penalised")
    opp_profit,   opp_soc   = optimize(df_ts, L, mode="opportunistic")

    return df_ts, base_profit, pen_profit, opp_profit, base_soc, pen_soc, opp_soc, gradcam

In [44]:
print("Checking data...")

print("merged_df_clean type:", type(merged_df_clean))
print("merged_df_spike type:", type(merged_df_spike))

if merged_df_clean is None:
    raise ValueError("merged_df_clean is None")

if merged_df_spike is None:
    raise ValueError("merged_df_spike is None")

print("Clean shape:", merged_df_clean.shape)
print("Spike shape:", merged_df_spike.shape)

print("\nSample data:")
print(merged_df_clean.head())

Checking data...
merged_df_clean type: <class 'pandas.core.frame.DataFrame'>
merged_df_spike type: <class 'pandas.core.frame.DataFrame'>
Clean shape: (1584, 9)
Spike shape: (1584, 9)

Sample data:
             datetime              node       SP15  NonSpin  RegDown  \
0 2023-01-09 08:00:00  TH_NP15_GEN-APND  136.59256     0.12     4.33   
1 2023-01-09 08:00:00  TH_SP15_GEN-APND  143.25455     0.12     4.33   
2 2023-01-09 08:00:00  TH_ZP26_GEN-APND  137.26883     0.12     4.33   
3 2023-01-09 09:00:00  TH_NP15_GEN-APND  134.78413     0.16     8.25   
4 2023-01-09 09:00:00  TH_SP15_GEN-APND  140.34950     0.16     8.25   

   Regulation Mileage Down  Regulation Mileage Up  RegUp  Spin  
0                      0.0                    0.0   5.99   1.5  
1                      0.0                    0.0   5.99   1.5  
2                      0.0                    0.0   5.99   1.5  
3                      0.0                    0.0   5.85   1.5  
4                      0.0                   

In [45]:
print("Running CLEAN scenario...")
clean_df, c_base, c_pen, c_opp, c_soc_base, c_soc_pen, c_soc_opp, c_gradcam = run(merged_df_clean)

print("Running ATTACK scenario...")
attack_df, a_base, a_pen, a_opp, a_soc_base, a_soc_pen, a_soc_opp, a_gradcam = run(merged_df_spike)

print("\n===== RESULTS =====")
print(f"{'Strategy':<20} {'Clean':>12} {'Attack':>12}")
print(f"{'Baseline':<20} {c_base:>12.2f} {a_base:>12.2f}")
print(f"{'Penalised':<20} {c_pen:>12.2f} {a_pen:>12.2f}")
print(f"{'Opportunistic':<20} {c_opp:>12.2f} {a_opp:>12.2f}")

print(f"\nAttack improvement (opp vs base): {((a_opp - a_base) / abs(a_base)) * 100:.2f}%")
print(f"Clean overhead (opp vs base):     {((c_opp - c_base) / abs(c_base)) * 100:.2f}%")

Running CLEAN scenario...
[VAE] Epoch 10/50 — Loss: 0.6566
[VAE] Epoch 20/50 — Loss: 0.4472
[VAE] Epoch 30/50 — Loss: 0.4031
[VAE] Epoch 40/50 — Loss: 0.4030
[VAE] Epoch 50/50 — Loss: 0.3725
Running ATTACK scenario...
[VAE] Epoch 10/50 — Loss: 1.4710
[VAE] Epoch 20/50 — Loss: 0.7650
[VAE] Epoch 30/50 — Loss: 0.5221
[VAE] Epoch 40/50 — Loss: 0.4781
[VAE] Epoch 50/50 — Loss: 0.4287

===== RESULTS =====
Strategy                    Clean       Attack
Baseline                 10414.99     10416.15
Penalised                 9218.46      9252.04
Opportunistic            12034.57     12171.77

Attack improvement (opp vs base): 16.85%
Clean overhead (opp vs base):     15.55%


In [46]:
# =========================
# 🔹 SAVE RESULTS
# =========================

import os

RESULT_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)


# ---- 1. Compute volatility FIRST before saving processed data ----
# Ensure run outputs exist before using them in save/export
if "clean_df" not in globals() or "c_base" not in globals():
    clean_df, c_base, c_anom, c_soc_base, c_soc_anom, c_gradcam = run(merged_df_clean)

if "attack_df" not in globals() or "a_base" not in globals():
    attack_df, a_base, a_anom, a_soc_base, a_soc_anom, a_gradcam = run(merged_df_spike)

clean_df["volatility"] = clean_df["SP15"].rolling(24).std()
attack_df["volatility"] = attack_df["SP15"].rolling(24).std()


# ---- 2. Save Profit Summary ----
results_df = pd.DataFrame({
    "scenario": ["clean_base", "clean_anom", "attack_base", "attack_anom"],
    "profit":   [c_base, c_anom, a_base, a_anom]
})
results_df.to_csv(os.path.join(RESULT_DIR, "results_summary.csv"), index=False)


# ---- 3. Save SOC (Clean) ----
soc_clean_df = pd.DataFrame({
    "baseline_soc": c_soc_base,
    "anomaly_soc":  c_soc_anom
})
soc_clean_df.to_csv(os.path.join(RESULT_DIR, "soc_clean.csv"), index=False)


# ---- 4. Save SOC (Attack) ----
soc_attack_df = pd.DataFrame({
    "baseline_soc": a_soc_base,
    "anomaly_soc":  a_soc_anom
})
soc_attack_df.to_csv(os.path.join(RESULT_DIR, "soc_attack.csv"), index=False)


# ---- 5. Save Processed Data (volatility + gradcam included) ----
clean_df.to_csv(os.path.join(RESULT_DIR, "clean_processed.csv"), index=False)
attack_df.to_csv(os.path.join(RESULT_DIR, "attack_processed.csv"), index=False)


# ---- 6. Save Volatility (standalone) ----
clean_df[["datetime", "volatility"]].to_csv(
    os.path.join(RESULT_DIR, "clean_volatility.csv"), index=False
)
attack_df[["datetime", "volatility"]].to_csv(
    os.path.join(RESULT_DIR, "attack_volatility.csv"), index=False
)


# ---- 7. Save Grad-CAM scores (standalone) ----
pd.DataFrame({
    "timestep": range(len(c_gradcam)),
    "gradcam":  c_gradcam
}).to_csv(os.path.join(RESULT_DIR, "clean_gradcam.csv"), index=False)

pd.DataFrame({
    "timestep": range(len(a_gradcam)),
    "gradcam":  a_gradcam
}).to_csv(os.path.join(RESULT_DIR, "attack_gradcam.csv"), index=False)


print(f"[SAVE] All results saved to: {RESULT_DIR}")

[SAVE] All results saved to: d:\battery-storage-optimization-energy-ancillary\results
